In [12]:
import sys
import os
sys.path.append(os.path.abspath('../../'))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from functions.preproc import *
from functions.ensembles import *

from sklearn import set_config
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import make_scorer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import StratifiedKFold
from sksurv.metrics import integrated_brier_score
from sksurv.metrics import concordance_index_ipcw
from sksurv.nonparametric import kaplan_meier_estimator
from sklearn.inspection import permutation_importance

from sksurv.ensemble import RandomSurvivalForest
from sksurv.metrics import concordance_index_censored
from sksurv.util import Surv

import lightgbm as lgb
from lightgbm import LGBMRegressor

set_config(display="text")  # displays text representation of estimators
import time

In [13]:
dataset = pd.read_csv("../data/data_final.csv")

diagnosis_variables = []

with open("../data/diagnosis_final.txt", "r") as file:
    for line in file:
        diagnosis_variables.append(line.strip())

treatment_variables = []
with open("../data/treatment_final.txt", "r") as file:
    for line in file:
        treatment_variables.append(line.strip())

survival_dataset = dataset.drop('Unnamed: 0', axis=1)

feature_columns = treatment_variables + diagnosis_variables

In [14]:
# Find duplicated patients
mask = (survival_dataset.duplicated(subset='PERSON_ID', keep=False)) #& (survival_dataset['Regional LN recurrence'] == 1)

# For each duplicated PERSON_ID with recurrence, keep only the row with the lowest survival days
to_remove = (
    survival_dataset[mask]
    .sort_values(['PERSON_ID', 'Survival days'], ascending=[True, True])
    .groupby('PERSON_ID')
    .head(-1)
    .index
)

survival_dataset = survival_dataset.drop(to_remove)

In [15]:
# Prepare data
data_x = survival_dataset[feature_columns]
data_y = Surv.from_arrays(
    event=survival_dataset['Regional LN recurrence'],
    time=survival_dataset['Survival days']
)

In [16]:
sgkf = StratifiedKFold(n_splits=5, shuffle=True, random_state=12)
cv = sgkf.split(X=data_x, y=survival_dataset['Regional LN recurrence'])


train_index, test_index = next(cv)
x_train = data_x.iloc[train_index]
y_train = data_y[train_index]

x_test = data_x.iloc[test_index]
y_test = data_y[test_index]

subsampling = survival_dataset.iloc[train_index].copy()

In [17]:
for train_idx, val_idx in cv:
    print("Train events:", survival_dataset.iloc[train_idx]["Regional LN recurrence"].sum())
    print("Val events:", survival_dataset.iloc[val_idx]["Regional LN recurrence"].sum())

Train events: 27.0
Val events: 6.0
Train events: 26.0
Val events: 7.0
Train events: 26.0
Val events: 7.0
Train events: 26.0
Val events: 7.0


In [18]:
# LightGBM Survival Model Wrapper
class LGBMSurvival:
    """
    LightGBM wrapper for survival analysis.
    Uses AFT (Accelerated Failure Time) model approach.
    """
    def __init__(self, learning_rate=0.1, n_estimators=100, max_depth=-1, 
                 num_leaves=31, min_data_in_leaf=20, random_state=None):
        self.learning_rate = learning_rate
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.num_leaves = num_leaves
        self.min_data_in_leaf = min_data_in_leaf
        self.random_state = random_state
        self.model = None
        self.y_train = None  # Store for baseline hazard estimation
        
    def fit(self, X, y):
        """
        Fit LightGBM model.
        y: structured array with 'event' and 'time' fields
        """
        # Store training data for baseline hazard
        self.y_train = y
        
        # Use log(time) as target for AFT model
        # This is a common approach for survival analysis with gradient boosting
        y_target = np.log(y['time'] + 1)  # Add 1 to avoid log(0)
        
        # Create sample weights: higher weight for events
        sample_weights = np.where(y['event'], 2.0, 1.0)
        
        self.model = LGBMRegressor(
            learning_rate=self.learning_rate,
            n_estimators=self.n_estimators,
            max_depth=self.max_depth,
            num_leaves=self.num_leaves,
            min_data_in_leaf=self.min_data_in_leaf,
            random_state=self.random_state,
            objective='regression',
            verbose=-1
        )
        
        self.model.fit(X, y_target, sample_weight=sample_weights)
        return self
    
    def predict(self, X):
        """
        Predict risk scores (higher = higher risk, lower = lower risk)
        For AFT model, we want to invert predictions: lower predicted time = higher risk
        """
        # Predict log(time)
        pred_log_time = self.model.predict(X)
        # Convert to risk score: negative of predicted time gives us risk
        # (lower predicted survival time = higher risk)
        risk_scores = -pred_log_time
        return risk_scores
    
    def score(self, X, y):
        """
        Calculate concordance index
        """
        predictions = self.predict(X)
        return concordance_index_censored(y['event'], y['time'], predictions)[0]
    
    def predict_survival_function(self, X, times=None):
        """
        Approximate survival function
        """
        # Predict log time
        pred_log_time = self.model.predict(X)
        pred_time = np.exp(pred_log_time)
        
        # Create survival functions
        surv_funcs = []
        for p_time in pred_time:
            # Use Weibull-like survival function
            # S(t) = exp(-(t/predicted_time)^shape)
            # Using shape=1 gives exponential distribution
            def make_surv_func(pt):
                return lambda t: np.exp(-np.maximum(0, t / pt))
            surv_funcs.append(make_surv_func(p_time))
        return surv_funcs

In [19]:
param_grid = {
    'learning_rate': [0.01, 0.05, 0.1],
    'n_estimators': [100, 200, 500],
    'max_depth': [3, 5, 7],
    'num_leaves': [31, 50, 100],
    'min_data_in_leaf': [10, 20, 50],
}

sgkf_1 = StratifiedKFold(n_splits=4, shuffle=True, random_state=12)
cv = sgkf_1.split(X=x_train, y=y_train['event'])

In [20]:
results_df, best_params, summary = generic_grid_search(
    LGBMSurvival,
    x_train,
    y_train,
    cv,
    param_grid,
    metrics=['c-index', 'c-index-ipcw', 'int-brier-score'],
    model_name='LightGBM',
    scale_data=False,
    scaler_type='standard',
    verbose=True,
    random_state=12,
    n_jobs=-1
)


LIGHTGBM GRID SEARCH (Manual Implementation)
Model: LGBMSurvival
Scaling: No
Subsampling: No
Metrics: ['c-index', 'c-index-ipcw', 'int-brier-score']
Parameter Grid:
  - learning_rate: [0.01, 0.05, 0.1]
  - n_estimators: [100, 200, 500]
  - max_depth: [3, 5, 7]
  - num_leaves: [31, 50, 100]
  - min_data_in_leaf: [10, 20, 50]
Total combinations: 243

[1/243] Testing params: {'learning_rate': 0.01, 'n_estimators': 100, 'max_depth': 3, 'num_leaves': 31, 'min_data_in_leaf': 10}
  c-index: 0.5118 ± 0.1973
  c-index-ipcw: 0.3934 ± 0.1339
  int-brier-score: 0.4190 ± 0.0116
[2/243] Testing params: {'learning_rate': 0.01, 'n_estimators': 100, 'max_depth': 3, 'num_leaves': 31, 'min_data_in_leaf': 20}
  Elapsed: 0.0min, Est. remaining: 1.6min
  c-index: 0.5369 ± 0.2047
  c-index-ipcw: 0.4016 ± 0.1527
  int-brier-score: 0.4192 ± 0.0108
[3/243] Testing params: {'learning_rate': 0.01, 'n_estimators': 100, 'max_depth': 3, 'num_leaves': 31, 'min_data_in_leaf': 50}
  Elapsed: 0.0min, Est. remaining: 1.

In [21]:
# View the results DataFrame
results_df.sort_values(by='c-index_mean', ascending=False)

,learning_rate,n_estimators,max_depth,num_leaves,min_data_in_leaf,c-index_mean,c-index_std,c-index-ipcw_mean,c-index-ipcw_std,int-brier-score_mean,...,c-index-ipcw_fold_2,c-index-ipcw_fold_3,c-index-ipcw_fold_4,int-brier-score_fold_1,int-brier-score_fold_2,int-brier-score_fold_3,int-brier-score_fold_4,c-index_has_error,c-index-ipcw_has_error,int-brier-score_has_error
19,0.01,100,7,31,20,0.618856,0.187002,0.497485,0.146169,0.408831,...,0.275221,0.606924,0.648022,0.413625,0.413199,0.391773,0.416725,False,False,False
25,0.01,100,7,100,20,0.618812,0.184870,0.513515,0.140968,0.408769,...,0.285934,0.615058,0.645240,0.413629,0.413179,0.391525,0.416744,False,False,False
22,0.01,100,7,50,20,0.618812,0.184870,0.513515,0.140968,0.408769,...,0.285934,0.615058,0.645240,0.413629,0.413179,0.391525,0.416744,False,False,False
73,0.01,500,7,31,20,0.609006,0.164470,0.513385,0.123516,0.388784,...,0.326118,0.590341,0.652056,0.398548,0.389839,0.372563,0.394186,False,False,False
49,0.01,200,7,50,20,0.605055,0.185015,0.498860,0.138223,0.398239,...,0.281306,0.578136,0.649386,0.405168,0.402528,0.380363,0.404896,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
108,0.05,200,3,31,10,0.503208,0.151544,0.449141,0.126385,0.391292,...,0.244098,0.451984,0.523784,0.400666,0.394853,0.377235,0.392413,False,False,False
114,0.05,200,3,100,10,0.503208,0.151544,0.449141,0.126385,0.391292,...,0.244098,0.451984,0.523784,0.400666,0.394853,0.377235,0.392413,False,False,False
162,0.10,100,3,31,10,0.488059,0.165737,0.456180,0.141253,0.391599,...,0.221206,0.504142,0.500639,0.402937,0.394260,0.377524,0.391675,False,False,False
165,0.10,100,3,50,10,0.488059,0.165737,0.456180,0.141253,0.391599,...,0.221206,0.504142,0.500639,0.402937,0.394260,0.377524,0.391675,False,False,False


In [22]:
results_df.sort_values(by='c-index_mean', ascending=False).to_csv("../results/lightgbm_grid_search_results.csv", index=False)